<a href="https://colab.research.google.com/github/rajeshsharma38/neural-network-learning/blob/main/MINST_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

In [2]:
import torch.nn.functional as F

In [3]:
df = pd.read_csv('mnist_train.csv', header=None)
df

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59997,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59998,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
df.shape
Ytr = torch.tensor(df.iloc[:, 0].values, dtype=torch.long)
Xtr = torch.tensor(df.iloc[:, 1:].values, dtype=torch.float32)
Xtr.shape, Ytr.shape, Xtr.dtype, Ytr.dtype

(torch.Size([60000, 784]), torch.Size([60000]), torch.float32, torch.int64)

In [13]:
#check kamming_normal initialisation
w1 = torch.randn(784, 100) / 784**0.5
b1 = torch.randn(100) * 0.01
w2 = torch.randn(100, 10) / 100**0.5
b2 = torch.randn(10) * 0

parameters = [w1, b1, w2, b2]

for p in parameters:
  p.requires_grad = True


In [14]:
print(sum(p.numel() for p in parameters))

79510


In [15]:
X_mean = Xtr.mean(0, keepdim=True)
X_std = Xtr.std(0, keepdim=True)
eps = 0.01

In [16]:
#forward pass
batch_size = 50
for i in range(40000): # Reduced iterations for faster debugging
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb = Xtr[ix]
  Yb = Ytr[ix]

  batch_norm = torch.nn.BatchNorm1d(num_features=784, momentum=0.01)
  Xb = batch_norm(Xb)

  # Xb_mean = Xb.mean(0, keepdim=True)
  # Xb_std = Xb.std(0, keepdim=True)

  # X_mean = 0.998 * X_mean + 0.002 * Xb_mean
  # X_std = 0.998 * X_std + 0.002 * Xb_std

  # Xb = (Xb - Xb_mean) / (Xb_std + eps)

  hpreact = Xb @ w1 + b1

  # Detach, convert to numpy, flatten, and then remove NaNs before plotting
  # import numpy as np # Import numpy if not already imported
  # hpreact_flat = hpreact.detach().numpy().flatten()
  # hpreact_flat = hpreact_flat[~np.isnan(hpreact_flat)] # Filter out NaN values

  # if hpreact_flat.size > 0: # Only plot if there's data after removing NaNs
  #   plt.hist(hpreact_flat, bins=200) # Plot a histogram of flattened hpreact values
  #   plt.title('Distribution of hpreact values (NaNs removed)') # Add a title
  #   plt.xlabel('Value') # Add x-axis label
  #   plt.ylabel('Frequency') # Add y-axis label
  #   plt.show() # Ensure the plot is displayed
  # else:
  #   print("No valid data to plot after removing NaNs from hpreact.")

  # break # The break statement is currently preventing the training loop from running

  h = torch.tanh(hpreact)

  logits = h @ w2 + b2
  loss = F.cross_entropy(logits, Yb)

  #print(loss.item())

  for p in parameters:
    p.grad = None

  loss.backward()

  lr = 0.1 if i < 20000 else 0.01

  for p in parameters:
    p.data += -lr * p.grad

  if i%100 == 0: # Print loss more frequently with fewer iterations
    print(loss.item())

2.4368066787719727
0.2595738470554352
0.7471044063568115
0.16832928359508514
0.27191057801246643
0.21179363131523132
0.21064653992652893
0.10647069662809372
0.22802399098873138
0.4708336293697357
0.23172391951084137
0.24017411470413208
0.18467730283737183
0.1421150416135788
0.18951302766799927
0.18676698207855225
0.14862699806690216
0.39344853162765503
0.15949447453022003
0.14354445040225983
0.10967505723237991
0.10049606114625931
0.12513583898544312
0.18853765726089478
0.06129583716392517
0.08096344023942947
0.03927011042833328
0.13751165568828583
0.13078831136226654
0.19741277396678925
0.027346882969141006
0.20649662613868713
0.1045018881559372
0.11782125383615494
0.13789400458335876
0.06798885017633438
0.09266195446252823
0.083370640873909
0.12435636669397354
0.14210030436515808
0.1139831468462944
0.036508433520793915
0.017201460897922516
0.19114315509796143
0.02732071653008461
0.20855756103992462
0.24603743851184845
0.10004313290119171
0.0629090815782547
0.028636258095502853
0.0952

Now that you've trained the model, let's use it to predict digits on a small portion of the test set (`mnist_test.csv`). We'll take the first 10 examples to quickly see how it performs.

In [17]:
# Load the test dataset (mnist_test.csv)
df_test = pd.read_csv('mnist_test.csv', header=None)

# Take only the first 10 examples for prediction
X_test = torch.tensor(df_test.iloc[:, 1:].values, dtype=torch.float32)
Y_test = torch.tensor(df_test.iloc[:, 0].values, dtype=torch.long)

print(f"Test data shape (features): {X_test.shape}")
print(f"Test data shape (labels): {Y_test.shape}")

Test data shape (features): torch.Size([10000, 784])
Test data shape (labels): torch.Size([10000])


In [18]:
X_test = batch_norm(X_test)

In [19]:
# Perform forward pass on the test data using the trained weights (w1, b1, w2, b2)
hpreact_test = X_test @ w1 + b1
h_test = torch.tanh(hpreact_test)
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    if predictions[i].item() == Y_test[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")

--- Predictions vs Actual Labels (First 10 Examples) ---
Matched: 9689 
 Unmatched: 311


In [60]:
X_tr1 = (Xtr - X_mean) / (X_std + eps)

In [21]:
hpreact_test = Xtr @ w1 + b1
h_test = torch.tanh(hpreact_test)
logits_test = h_test @ w2 + b2

# Get the predicted class by finding the index of the maximum logit value
predictions = torch.argmax(logits_test, dim=1)

match_count = 0

print("--- Predictions vs Actual Labels (First 10 Examples) ---")
for i in range(len(predictions)):
    if predictions[i].item() == Ytr[i].item():
      match_count += 1
print(f"Matched: {match_count} \n Unmatched: {len(predictions) - match_count}")

--- Predictions vs Actual Labels (First 10 Examples) ---
Matched: 55654 
 Unmatched: 4346
